# What Do People Say When They Post About Disasters?
### DATASCI 530 – Project 3 · Xueqian Cheng

*Draft: sections 1 and 4–6 will be filled in once modeling is done.*

**Key findings** *(to come)*

## 1. Introduction

*(To come: topic, research question, and summary of findings.)*

**Research question (working).** Once the hashtags used to collect the posts are
removed, what themes appear in the captions of disaster-related social media
posts, and how closely do they match the image-based damage labels?

## 2. Data

The data are the **Multimodal Damage Identification for Humanitarian Computing**
dataset from the UCI Machine Learning Repository (Mouzannar, Rizk & Awad, 2018;
DOI 10.24432/C52P6P). The authors collected public Instagram and Twitter posts
from 2012 to 2017 by searching for disaster-related hashtags such as
`#buildingfire`, `#floodwater` or `#earthquake`, and for everyday hashtags such as
`#food` or `#ad` to get posts without damage. Each post has an image, its caption
and one of six labels, which the authors assigned **by looking at the image**:

| Label | Meaning |
|---|---|
| Infrastructure | Damaged buildings, roads, vehicles |
| Nature | Damaged landscape: drought, landslides, fallen trees |
| Fires | Building or forest fires |
| Flood | Flooded areas |
| Human | Injured or killed people |
| No damage | Everyday posts |

This project uses only the **caption text**. The file name of each caption
records the search hashtag and the date and time it was posted.

*License note: the dataset is for academic research only, and its images and
text may not be posted online. The raw data are kept out of version control, and
this report shows summary statistics rather than individual captions.*

## 3. Data preparation and quality checks

In [ ]:
import re
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option("display.precision", 2)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})

RAW_ZIP   = Path("data_raw/damage.zip")
CLEAN_DIR = Path("data_clean")

# Short display names and one fixed color per label, used in every chart
LABELS = {"damaged_infrastructure": "Infrastructure", "damaged_nature": "Nature",
          "fires": "Fires", "flood": "Flood", "human_damage": "Human",
          "non_damage": "No damage"}
LABEL_ORDER = list(LABELS.values())
LABEL_COLOR = dict(zip(LABEL_ORDER, ["#4e79a7", "#59a14f", "#e15759",
                                     "#76b7b2", "#b07aa1", "#bab0ac"]))

**Load the captions.** The captions are read straight from the downloaded zip
file (the images are not needed). The label comes from the folder, and the
search hashtag and date come from the file name, e.g.
`flood/text/floodwater_2017-09-01_14-03-22.txt`.

In [ ]:
FILE_PATTERN = re.compile(r"multimodal/(\w+)/text/(\w+?)_(\d{4}-\d{2}-\d{2})_[\d-]+\.txt$")

rows = []
with zipfile.ZipFile(RAW_ZIP) as z:
    for name in z.namelist():
        match = FILE_PATTERN.match(name)
        if match:
            label, query, date = match.groups()
            rows.append({"label": LABELS[label], "query": query.lower(),
                         "date": pd.to_datetime(date),
                         "text": z.read(name).decode("utf-8", errors="replace")})

posts = pd.DataFrame(rows)
posts["label"] = pd.Categorical(posts["label"], LABEL_ORDER, ordered=True)
print(f"{len(posts):,} captions loaded")

### 3.1 Quality checks on the raw captions

In [ ]:
ARABIC = r"[؀-ۿ]"

def arabic_share(text):
    """Share of word characters in a caption that are Arabic script."""
    letters = re.findall(r"\w", text)
    return len(re.findall(ARABIC, text)) / max(len(letters), 1)

posts["norm_text"]    = posts["text"].str.lower().str.split().str.join(" ")
posts["n_words_raw"]  = posts["text"].str.split().str.len()
posts["is_duplicate"] = posts["norm_text"].duplicated(keep=False)
posts["is_arabic"]    = posts["text"].map(arabic_share) > 0.5
posts["has_own_query"] = [f"#{q}" in t for q, t in zip(posts["query"], posts["norm_text"])]

checks = (posts.groupby("label", observed=True)
               .rename_axis(None)
               .agg(n=("text", "size"),
                    median_words=("n_words_raw", "median"),
                    duplicate=("is_duplicate", "mean"),
                    arabic=("is_arabic", "mean"),
                    own_query=("has_own_query", "mean")))
checks.loc["All posts"] = [len(posts), posts["n_words_raw"].median(),
                           posts["is_duplicate"].mean(), posts["is_arabic"].mean(),
                           posts["has_own_query"].mean()]

display(checks.style
        .format({"n": "{:,.0f}", "median_words": "{:.0f}", "duplicate": "{:.0%}",
                 "arabic": "{:.0%}", "own_query": "{:.0%}"})
        .relabel_index(["Captions", "Median words", "Duplicated text",
                        "Mostly Arabic", "Contains its search hashtag"], axis=1)
        .set_caption("Table 1. Quality checks on the raw captions, by label"))
print(f"Date range: {posts['date'].min():%b %Y} to {posts['date'].max():%b %Y}; "
      f"{(posts['date'].dt.year == 2017).mean():.0%} of posts are from 2017")

**What it shows.**
- **No captions are empty, but they are short.** The median is under 30 words.
- **Some texts are repeated.** The same caption often appears with several
  images, e.g. a post with several photos.
- **A small share is in Arabic**, mostly infrastructure and human-damage posts
  from the wars in Syria and Yemen.
- **Most important: about 9 in 10 captions contain the hashtag used to find
  them.** A text model would then simply learn the search terms instead of what
  people write, so these hashtags must be removed before modeling (Section 3.2).
- **The posts are not spread over time.** Most are from 2017, so trends over
  time are not studied.

In [ ]:
counts = posts["label"].value_counts().reindex(LABEL_ORDER)

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.barh(counts.index[::-1], counts.values[::-1],
        color=[LABEL_COLOR[l] for l in counts.index[::-1]])
for y, n in enumerate(counts.values[::-1]):
    ax.text(n + 30, y, f"{n:,}", va="center")
ax.set_xlabel("Number of captions")
ax.set_title("Figure 1. Half of the posts show no damage", loc="left")
plt.tight_layout()
plt.show()

**What it shows.** The labels are unbalanced: half of all posts show no damage,
and human damage, the smallest group, has only 240 posts. Results that use the
labels (Section 5) will therefore be reported per label, not only overall.

**Which hashtags were used to collect each label?** The table lists the three
most common search hashtags per label and their share of the label's posts.

In [ ]:
def top_queries(group, k=3):
    shares = group.value_counts(normalize=True).head(k)
    return ", ".join(f"#{q} ({s:.0%})" for q, s in shares.items())

query_table = (posts.groupby("label", observed=True)["query"]
                    .agg(n_queries="nunique", top_queries=top_queries)
                    .rename_axis(None))
display(query_table.style
        .relabel_index(["Search hashtags", "Most common (share of posts)"], axis=1)
        .set_caption("Table 2. Search hashtags used to collect each label"))

**What it shows.** Each label was collected through a few hashtags. Two thirds of
the posts without damage come from a single search, `#ad`, so many of them are
advertisements. Some hashtags such as `#hurricaneirma` or `#disasters` were used
for several labels, which already shows that the text alone will not always
separate the labels.

### 3.2 Cleaning

The cleaning steps, in order:

1. **Duplicates.** A repeated caption is kept once. The few texts that appear
   under *different* labels are dropped, because their label is unclear.
2. **Arabic posts.** Captions that are mostly Arabic are dropped. The text
   methods used here rely on English word lists, so a few Arabic posts would form
   their own cluster because of their language, not their content.
3. **Text cleaning.** Links and `@user` mentions are removed. **Every search
   hashtag (all 37) is removed**, whichever label it was used for, so that the
   collection method does not leak into the model. Other hashtags carry real
   content (e.g. `#rescue`, `#prayfor...`), so only the `#` is removed and the word
   is kept. Normal words are also kept even when they match a search term:
   *"the earthquake hit at night"* stays as it is.
4. **Very short posts.** Captions with fewer than 3 words left are dropped.

In [ ]:
SEARCH_TAGS = set(posts["query"])

def clean_caption(text):
    """Lowercase a caption and remove links, mentions and search hashtags."""
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@[\w.]+", " ", text)
    text = re.sub(r"#(\w+)",
                  lambda m: " " if m[1].lower() in SEARCH_TAGS else f" {m[1]} ", text)
    return " ".join(text.lower().split())

steps = [("Raw captions", len(posts))]

# 1. Duplicates: drop texts with conflicting labels, then keep one copy
n_labels = posts.groupby("norm_text")["label"].transform("nunique")
clean = posts[n_labels == 1].drop_duplicates("norm_text")
steps.append(("Duplicates removed", len(clean)))

# 2. Arabic posts
clean = clean[~clean["is_arabic"]]
steps.append(("Mostly Arabic posts removed", len(clean)))

# 3–4. Clean the text and drop very short posts
clean = clean.assign(clean_text=clean["text"].map(clean_caption))
clean["n_words"] = clean["clean_text"].str.split().str.len()
clean = clean[clean["n_words"] >= 3].reset_index(drop=True)
steps.append(("Posts with fewer than 3 words removed", len(clean)))

funnel = pd.DataFrame(steps, columns=["Step", "Posts left"])
funnel["Removed"] = (-funnel["Posts left"].diff()).fillna(0)
display(funnel.style.format({"Posts left": "{:,}", "Removed": "{:,.0f}"})
        .hide(axis="index")
        .set_caption("Table 3. Posts kept after each cleaning step"))

**Check: how much of the search term is left?** The search hashtags are gone,
but a post found through `#flood` may still use the word *flood* in its text.
That is genuine content, but it means the labels can still partly be recovered
from the words, which should be kept in mind when comparing clusters to labels.

In [ ]:
has_term = [f" {q} " in f" {t} " for q, t in zip(clean["query"], clean["clean_text"])]
print(f"Before cleaning, {posts['has_own_query'].mean():.0%} of captions contained "
      f"their search hashtag; after cleaning, {pd.Series(has_term).mean():.0%} "
      "still use the search term as a normal word.")

In [ ]:
kept = clean.groupby("label", observed=True)
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.boxplot([g["n_words"] for _, g in kept], orientation="horizontal", showfliers=False,
           tick_labels=[l for l, _ in kept], patch_artist=True,
           boxprops={"facecolor": "#dfe7f2"}, medianprops={"color": "#1f3b63"})
ax.invert_yaxis()
ax.set_xlabel("Words per caption after cleaning (outliers hidden)")
ax.set_title("Figure 2. Infrastructure, nature and flood captions are the shortest", loc="left")
plt.tight_layout()
plt.show()

summary = kept.agg(posts=("clean_text", "size"), median_words=("n_words", "median"))
display(summary.rename_axis(None).T.style.format("{:,.0f}")
        .relabel_index(["Posts", "Median words"], axis=0)
        .set_caption("Table 4. The cleaned dataset, by label"))

**What it shows.** Cleaning removed 676 posts (12%), mostly repeated captions,
and left 5,155. Every label keeps at least 180 posts. Removing the search
hashtags worked: before cleaning 91% of captions contained their search hashtag,
and afterwards only 2% still use the search term as a normal word.
Infrastructure, nature and flood captions are the shortest (median around 20
words), while human-damage posts are longer, because they often tell a story
about war victims. Short texts carry few words
per post, which matters for the choice of method: classic topic models such as
LDA work best with longer documents, so Section 4 compares them with methods
designed for short texts.

In [ ]:
CLEAN_DIR.mkdir(exist_ok=True)
clean[["label", "query", "date", "clean_text", "n_words"]].to_csv(
    CLEAN_DIR / "captions_clean.csv", index=False)
print(f"Saved {len(clean):,} cleaned captions to {CLEAN_DIR / 'captions_clean.csv'}")

## 4. Methods

*(To come: TF-IDF + NMF and LDA topic models compared with sentence embeddings +
K-means; choosing the number of topics with coherence and silhouette scores;
stability across random seeds.)*

## 5. Results

*(To come.)*

## 6. Discussion

*(To come.)*